In [0]:
import time
import json
import requests
import pandas as pd

## Data Source: CQC Syndication API

For this Home/Domiciliary Care Market Analysis project, we used the **CQC Syndication API** to extract Kent-based 
domiciliary/personal care provider data.

**Base URL:** `https://api.service.cqc.org.uk/public/v1`

### Getting Access

1. Register for a free account at [api-portal.service.cqc.org.uk/signup](https://api-portal.service.cqc.org.uk/signup)
2. Subscribe to the **Syndication** product
3. Collect your subscription key from the portal's Profile page

In [0]:

BASE_URL = "https://api.service.cqc.org.uk/public/v1"

# Getting CQC subscription key / API key from the secrets in Databricks
CQC_SUBSCRIPTION_KEY = dbutils.secrets.get(
    catalog="domiciliarycare", schema="security", key="api_key"
)

HEADERS = {
    "Ocp-Apim-Subscription-Key": CQC_SUBSCRIPTION_KEY,
    "User-Agent": "HomeSafeKentPipeline/1.0",  
    "Accept": "application/json",
}
print("Confirmed - Key loaded from Unity Catalog secret.")

### API Response Handling

The `cqc_get()` function below is used to send requests to the CQC Syndication API and 
handle its response — including checking for and reacting appropriately to different 
status codes, rather than assuming every request succeeds.

Per CQC's own API documentation, requests to this endpoint can return the following 
status codes: 
- **200 (OK)**
- **400 (Bad Request)**
- **404 (Not Found)**
- **500 (Internal Server Error)**

We additionally handle :
- **401 (Unauthorized)** 
- **502 (Bad Gateway)** <br>
which are not listed in CQC endpoint documentation but occur at the API gateway level general practice. 

We also handle:
- **429 (Too Many Requests)** <br>
explicitly with a wait-and-retry mechanism, since CQC enforces a **rate limit of 100 requests per 5 seconds**. Rather than allowing the pipeline to fail when this limit is hit, our code pauses (using the `Retry-After` value 
CQC provides, or a 5-second default) and automatically retries — up to 3 attempts per 
request , so a single burst of rate limiting doesn't crash the full extraction run across hundreds of records.

In [0]:
def cqc_get(path, params=None, max_retries=3):
    url = f"{BASE_URL}{path}"
    resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code == 404:
            print(f"404 Not Found - no record exists at {url}")
            return None

        if resp.status_code == 400:
            raise RuntimeError("400 Bad Request - check your request parameters.")

        if resp.status_code == 401:
            raise RuntimeError("401 Unauthorized - key missing or invalid.")

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited (429). Waiting {wait}s before retry {attempt}/{max_retries}...")
            time.sleep(wait)
            continue

        if resp.status_code in (500, 502):
            raise RuntimeError(f"{resp.status_code} -- CQC server-side error, not a request problem.")

        resp.raise_for_status()

    raise RuntimeError(f"Gave up after {max_retries} attempts (last status {resp.status_code if resp else 'n/a'}).")

### Fetching Location Data via the CQC API

We now fetch location data through the CQC Syndication API, filtered by:

- **Local Authority** - `Kent`
- **Regulated Activity** - `Personal care`

We filter specifically on **Personal care** as Home Safe is only interested in the **home care / domiciliary care** 
market — services delivered to a person in their own home. 

In [0]:

def fetch_kent_data(local_authority="Kent", regulated_activity="Personal care", per_page=1000, polite_delay=0.3):
    """Page through /locations filtered to Kent + Personal care."""
    params = [
        ("localAuthority", local_authority),
        ("regulatedActivity", regulated_activity),
        ("perPage", per_page),
    ]
    all_locations = []
    page = 1
    while True:
        page_params = params + [("page", page)]
        data = cqc_get("/locations", params=page_params)
        all_locations.extend(data["locations"])
        print(f"Page {data['page']}/{data['totalPages']} -- {len(data['locations'])} rows "
              f"(running total {len(all_locations)})")
        if not data.get("nextPageUri"):
            break
        page += 1
        time.sleep(polite_delay)
    return all_locations

kent_locations_summary = fetch_kent_data()
print(f"\nTotal Kent personal-care locations: {len(kent_locations_summary)}")


In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")

detail_records = []
failed_ids = []
for i, loc in enumerate(kent_locations_summary, 1):
    try:
        detail_records.append(fetch_location_detail(loc["locationId"]))
    except Exception as e:
        print(f"FAILED on {loc['locationId']} ({loc.get('locationName')}): {e}")
        failed_ids.append(loc["locationId"])
    if i % 50 == 0:
        print(f"Processed {i}/{len(kent_locations_summary)}...")
    time.sleep(0.3)

print(f"\nDone. Fetched {len(detail_records)} records. Failed: {len(failed_ids)}")
if failed_ids:
    print(f"Failed IDs: {failed_ids}")

In [0]:
def flatten_location_full(rec):
    """Every scalar/flat field from the location record -- the main table."""
    return {
        "location_id": rec.get("locationId"),
        "provider_id": rec.get("providerId"),
        "organisation_type": rec.get("organisationType"),
        "type": rec.get("type"),
        "name": rec.get("name"),
        "onspd_ccg_code": rec.get("onspdCcgCode"),
        "onspd_ccg_name": rec.get("onspdCcgName"),
        "onspd_icb_code": rec.get("onspdIcbCode"),
        "onspd_icb_name": rec.get("onspdIcbName"),
        "ods_code": rec.get("odsCode"),
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "dormancy": rec.get("dormancy"),
        "latitude": rec.get("onspdLatitude"),
        "longitude": rec.get("onspdLongitude"),
        "care_home": rec.get("careHome"),
        "inspection_directorate": rec.get("inspectionDirectorate"),
        "postal_address_line1": rec.get("postalAddressLine1"),
        "postal_address_line2": rec.get("postalAddressLine2"),
        "postal_address_town_city": rec.get("postalAddressTownCity"),
        "region": rec.get("region"),
        "postal_code": rec.get("postalCode"),
        "uprn": rec.get("uprn"),
        "main_phone_number": rec.get("mainPhoneNumber"),
        "website": rec.get("website"),
        "number_of_beds": rec.get("numberOfBeds"),
        "constituency": rec.get("constituency"),
        "local_authority": rec.get("localAuthority"),
        "last_inspection_date": (rec.get("lastInspection") or {}).get("date"),
        "last_report_publication_date": (rec.get("lastReport") or {}).get("publicationDate"),
    }
    

In [0]:
def extract_regulated_activities(rec):
    """One row per (location, regulated activity). Includes contacts flattened too."""
    rows = []
    location_id = rec.get("locationId")
    for activity in rec.get("regulatedActivities", []) or []:
        contacts = activity.get("contacts", []) or []
        if contacts:
            for contact in contacts:
                rows.append({
                    "location_id": location_id,
                    "activity_name": activity.get("name"),
                    "activity_code": activity.get("code"),
                    "contact_title": contact.get("personTitle"),
                    "contact_given_name": contact.get("personGivenName"),
                    "contact_family_name": contact.get("personFamilyName"),
                    "contact_roles": "; ".join(contact.get("personRoles", []) or []),
                })
        else:
            rows.append({
                "location_id": location_id,
                "activity_name": activity.get("name"),
                "activity_code": activity.get("code"),
                "contact_title": None,
                "contact_given_name": None,
                "contact_family_name": None,
                "contact_roles": None,
            })
    return rows

In [0]:
def extract_gac_service_types(rec):
    location_id = rec.get("locationId")
    return [
        {
            "location_id": location_id,
            "service_type_name": s.get("name"),
            "service_type_description": s.get("description"),
        }
        for s in rec.get("gacServiceTypes", []) or []
    ]

In [0]:
def extract_specialisms(rec):
    location_id = rec.get("locationId")
    return [
        {"location_id": location_id, "specialism_name": s.get("name")}
        for s in rec.get("specialisms", []) or []
    ]

In [0]:
def extract_inspection_categories(rec):
    location_id = rec.get("locationId")
    return [
        {
            "location_id": location_id,
            "category_code": c.get("code"),
            "is_primary": c.get("primary"),
            "category_name": c.get("name"),
        }
        for c in rec.get("inspectionCategories", []) or []
    ]

In [0]:
def extract_inspection_areas(rec):
    location_id = rec.get("locationId")
    return [
        {
            "location_id": location_id,
            "inspection_area_id": a.get("inspectionAreaId"),
            "inspection_area_name": a.get("inspectionAreaName"),
            "status": a.get("status"),
            "end_date": a.get("endDate"),
            "superseded_by": "; ".join(a.get("supersededBy", []) or []),
        }
        for a in rec.get("inspectionAreas", []) or []
    ]

In [0]:
def extract_reports(rec):
    location_id = rec.get("locationId")
    return [
        {
            "location_id": location_id,
            "report_link_id": r.get("linkId"),
            "report_date": r.get("reportDate"),
            "first_visit_date": r.get("firstVisitDate"),
            "report_uri": r.get("reportUri"),
            "report_type": r.get("reportType"),
        }
        for r in rec.get("reports", []) or []
    ]

In [0]:
def extract_ratings(rec):
    location_id = rec.get("locationId")
    ratings = (rec.get("currentRatings") or {}).get("overall") or {}
    rows = []
    for kq in ratings.get("keyQuestionRatings", []) or []:
        rows.append({
            "location_id": location_id,
            "question_name": kq.get("name"),
            "rating": kq.get("rating"),
            "report_date": kq.get("reportDate"),
            "report_link_id": kq.get("reportLinkId"),
        })
    return rows

In [0]:
# Main table
main_df = pd.DataFrame([flatten_location_full(r) for r in detail_records])

# Related tables -- flatten the list-of-lists into one flat list per table
regulated_activities_df = pd.DataFrame(
    [row for r in detail_records for row in extract_regulated_activities(r)]
)
gac_service_types_df = pd.DataFrame(
    [row for r in detail_records for row in extract_gac_service_types(r)]
)
specialisms_df = pd.DataFrame(
    [row for r in detail_records for row in extract_specialisms(r)]
)
inspection_categories_df = pd.DataFrame(
    [row for r in detail_records for row in extract_inspection_categories(r)]
)
inspection_areas_df = pd.DataFrame(
    [row for r in detail_records for row in extract_inspection_areas(r)]
)
reports_df = pd.DataFrame(
    [row for r in detail_records for row in extract_reports(r)]
)
ratings_df = pd.DataFrame(
    [row for r in detail_records for row in extract_ratings(r)]
)

print(f"main_df: {len(main_df)} rows")
print(f"regulated_activities_df: {len(regulated_activities_df)} rows")
print(f"gac_service_types_df: {len(gac_service_types_df)} rows")
print(f"specialisms_df: {len(specialisms_df)} rows")
print(f"inspection_categories_df: {len(inspection_categories_df)} rows")
print(f"inspection_areas_df: {len(inspection_areas_df)} rows")
print(f"reports_df: {len(reports_df)} rows")
print(f"ratings_df: {len(ratings_df)} rows")

In [0]:
tables = {
    "cqc_kent_locations_main": main_df,
    "cqc_kent_regulated_activities": regulated_activities_df,
    "cqc_kent_gac_service_types": gac_service_types_df,
    "cqc_kent_specialisms": specialisms_df,
    "cqc_kent_inspection_categories": inspection_categories_df,
    "cqc_kent_reports": reports_df,
    "cqc_kent_ratings": ratings_df,
}

for table_name, df in tables.items():
    spark_df = spark.createDataFrame(df.astype(str))
    spark_df.write.format("delta").mode("overwrite").saveAsTable(f"domiciliarycare.bronze.{table_name}")
    print(f"Saved {len(df)} rows to domiciliarycare.bronze.{table_name}")

In [0]:
kent_provider_ids = sorted({r.get("providerId") for r in detail_records if r.get("providerId")})
print(f"{len(kent_provider_ids)} distinct providers to fetch")

In [0]:
def fetch_provider_detail(provider_id):
    return cqc_get(f"/providers/{provider_id}")

provider_records = []
failed_provider_ids = []

for i, pid in enumerate(kent_provider_ids, 1):
    try:
        provider_records.append(fetch_provider_detail(pid))
    except Exception as e:
        print(f"FAILED on provider {pid}: {e}")
        failed_provider_ids.append(pid)
    if i % 20 == 0:
        print(f"Fetched {i}/{len(kent_provider_ids)} providers...")
    time.sleep(0.3)

print(f"\nDone. Fetched {len(provider_records)} provider records. Failed: {len(failed_provider_ids)}")

In [0]:
def flatten_provider_main(rec):
    return {
        "provider_id": rec.get("providerId"),
        "ownership_type": rec.get("ownershipType"),
        "type": rec.get("type"),
        "name": rec.get("name"),
        "brand_id": rec.get("brandId"),
        "brand_name": rec.get("brandName"),
        "ods_code": rec.get("odsCode"),
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "companies_house_number": rec.get("companiesHouseNumber"),
        "charity_number": rec.get("charityNumber"),
        "website": rec.get("website"),
        "postal_address_line1": rec.get("postalAddressLine1"),
        "postal_address_line2": rec.get("postalAddressLine2"),
        "postal_address_town_city": rec.get("postalAddressTownCity"),
        "postal_address_county": rec.get("postalAddressCounty"),
        "region": rec.get("region"),
        "postal_code": rec.get("postalCode"),
        "also_known_as": rec.get("alsoKnownAs"),
        "deregistration_date": rec.get("deregistrationDate"),
        "uprn": rec.get("uprn"),
        "latitude": rec.get("onspdLatitude"),
        "longitude": rec.get("onspdLongitude"),
        "onspd_icb_code": rec.get("onspdIcbCode"),
        "onspd_icb_name": rec.get("onspdIcbName"),
        "main_phone_number": rec.get("mainPhoneNumber"),
        "inspection_directorate": rec.get("inspectionDirectorate"),
        "constituency": rec.get("constituency"),
        "local_authority": rec.get("localAuthority"),
        "last_inspection_date": (rec.get("lastInspection") or {}).get("date"),
        "last_report_publication_date": (rec.get("lastReport") or {}).get("publicationDate"),
    }


def extract_provider_locations(rec):
    """Bridge table: which locations belong to this provider."""
    provider_id = rec.get("providerId")
    return [{"provider_id": provider_id, "location_id": lid} for lid in rec.get("locationIds", []) or []]


def extract_provider_contacts(rec):
    provider_id = rec.get("providerId")
    return [
        {
            "provider_id": provider_id,
            "contact_title": c.get("personTitle"),
            "contact_given_name": c.get("personGivenName"),
            "contact_family_name": c.get("personFamilyName"),
            "contact_roles": "; ".join(c.get("personRoles", []) or []),
        }
        for c in rec.get("contacts", []) or []
    ]


def extract_provider_relationships(rec):
    provider_id = rec.get("providerId")
    return [
        {
            "provider_id": provider_id,
            "related_provider_id": r.get("relatedProviderId"),
            "related_provider_name": r.get("relatedProviderName"),
            "relationship_type": r.get("type"),
            "reason": r.get("reason"),
        }
        for r in rec.get("relationships", []) or []
    ]


def extract_provider_regulated_activities(rec):
    provider_id = rec.get("providerId")
    rows = []
    for a in rec.get("regulatedActivities", []) or []:
        nominee = a.get("nominatedIndividual", {}) or {}
        rows.append({
            "provider_id": provider_id,
            "activity_name": a.get("name"),
            "activity_code": a.get("code"),
            "nominee_title": nominee.get("personTitle"),
            "nominee_given_name": nominee.get("personGivenName"),
            "nominee_family_name": nominee.get("personFamilyName"),
        })
    return rows


def extract_provider_inspection_categories(rec):
    provider_id = rec.get("providerId")
    return [
        {"provider_id": provider_id, "category_code": c.get("code"),
         "is_primary": c.get("primary"), "category_name": c.get("name")}
        for c in rec.get("inspectionCategories", []) or []
    ]


def extract_provider_inspection_areas(rec):
    provider_id = rec.get("providerId")
    return [
        {
            "provider_id": provider_id,
            "inspection_area_id": a.get("inspectionAreaId"),
            "inspection_area_name": a.get("inspectionAreaName"),
            "status": a.get("status"),
            "end_date": a.get("endDate"),
            "superseded_by": "; ".join(a.get("supersededBy", []) or []),
        }
        for a in rec.get("inspectionAreas", []) or []
    ]


def extract_provider_reports(rec):
    provider_id = rec.get("providerId")
    return [
        {
            "provider_id": provider_id,
            "report_link_id": r.get("linkId"),
            "report_date": r.get("reportDate"),
            "first_visit_date": r.get("firstVisitDate"),
            "report_uri": r.get("reportUri"),
            "report_type": r.get("reportType"),
        }
        for r in rec.get("reports", []) or []
    ]


def extract_provider_ratings(rec):
    """Overall + key question ratings, flattened."""
    provider_id = rec.get("providerId")
    overall = (rec.get("currentRatings") or {}).get("overall") or {}
    rows = []
    for kq in overall.get("keyQuestionRatings", []) or []:
        rows.append({
            "provider_id": provider_id,
            "rating_level": "overall_key_question",
            "question_name": kq.get("name"),
            "rating": kq.get("rating"),
            "report_date": kq.get("reportDate"),
        })
    # Service-level ratings (a provider can have multiple, e.g. hospitals with several departments)
    for sr in (rec.get("currentRatings") or {}).get("serviceRatings", []) or []:
        rows.append({
            "provider_id": provider_id,
            "rating_level": f"service:{sr.get('name')}",
            "question_name": "overall",
            "rating": sr.get("rating"),
            "report_date": sr.get("reportDate"),
        })
    return rows

In [0]:
main_provider_df = pd.DataFrame([flatten_provider_main(r) for r in provider_records])
provider_locations_df = pd.DataFrame([row for r in provider_records for row in extract_provider_locations(r)])
provider_contacts_df = pd.DataFrame([row for r in provider_records for row in extract_provider_contacts(r)])
provider_relationships_df = pd.DataFrame([row for r in provider_records for row in extract_provider_relationships(r)])
provider_activities_df = pd.DataFrame([row for r in provider_records for row in extract_provider_regulated_activities(r)])
provider_inspection_categories_df = pd.DataFrame([row for r in provider_records for row in extract_provider_inspection_categories(r)])
provider_reports_df = pd.DataFrame([row for r in provider_records for row in extract_provider_reports(r)])
provider_ratings_df = pd.DataFrame([row for r in provider_records for row in extract_provider_ratings(r)])

for name, df in {
    "cqc_kent_providers_main": main_provider_df,
    "cqc_kent_provider_locations_bridge": provider_locations_df,
    "cqc_kent_provider_contacts": provider_contacts_df,
    "cqc_kent_provider_relationships": provider_relationships_df,
    "cqc_kent_provider_regulated_activities": provider_activities_df,
    "cqc_kent_provider_inspection_categories": provider_inspection_categories_df,
    "cqc_kent_provider_reports": provider_reports_df,
    "cqc_kent_provider_ratings": provider_ratings_df,
}.items():
    if df.empty:
        print(f"Skipped {name} (no rows)")
        continue
    spark.createDataFrame(df.astype(str)).write.format("delta").mode("overwrite").saveAsTable(f"domiciliarycare.bronze.{name}")
    print(f"Saved {len(df)} rows to domiciliarycare.bronze.{name}")